<a href="https://colab.research.google.com/github/Thampi-hub/Springboard_RT/blob/main/Capstone_2/2_DataWrangling/readData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

In [2]:
#Read in JSON files (Category names):
# ca_json = pd.read_json("./Kaggle_data/CA_category_id.json")["items"].to_dict()
# us_json = pd.read_json("./Kaggle_data/US_category_id.json")["items"].to_dict()
#       --------- OR ---------
url_ca_json = "https://raw.githubusercontent.com/Thampi-hub/Springboard_RT/refs/heads/main/Capstone_2/2_DataWrangling/Kaggle_data/CA_category_id.json"
url_us_json = "https://raw.githubusercontent.com/Thampi-hub/Springboard_RT/refs/heads/main/Capstone_2/2_DataWrangling/Kaggle_data/US_category_id.json"
ca_json = pd.read_json(url_ca_json)["items"].to_dict()
us_json = pd.read_json(url_us_json)["items"].to_dict()

cat_id  = []
cat_val = []
for idx in ca_json.values():
    cat_id.append( int(idx["id"]) )
    cat_val.append(idx["snippet"]["title"])
for idx in us_json.values():
    cat_id.append( int(idx["id"]) )
    cat_val.append(idx["snippet"]["title"])

category_mapping = pd.DataFrame({'category_id':cat_id, 'category':cat_val }).drop_duplicates()

In [3]:
#Read in CSV files (Core data files):
# ca_csv = pd.read_csv("./Kaggle_data/CAvideos.csv")
# us_csv = pd.read_csv("./Kaggle_data/USvideos.csv")
#       --------- OR ---------
url_ca_csv = "https://raw.githubusercontent.com/Thampi-hub/Springboard_RT/refs/heads/main/Capstone_2/2_DataWrangling/Kaggle_data/CAvideos.csv"
url_us_csv = "https://raw.githubusercontent.com/Thampi-hub/Springboard_RT/refs/heads/main/Capstone_2/2_DataWrangling/Kaggle_data/USvideos.csv"
ca_csv = pd.read_csv(url_ca_csv)
us_csv = pd.read_csv(url_us_csv)

ca_csv['country'] = "Canada"
us_csv['country'] = "USA"

allCSV = pd.concat([ca_csv,us_csv], axis=0)
print("CSV dimensions: ", ca_csv.shape, " + ", us_csv.shape, " = ", allCSV.shape)


CSV dimensions:  (40881, 17)  +  (40949, 17)  =  (81830, 17)


In [73]:
#Merge all data:
allData = pd.merge(allCSV, category_mapping, on="category_id", how="left")

print("Dimensions: ", allData.shape, "\n\n")
print(allData.columns.values,"\n")
print(allData.dtypes)

Dimensions:  (81830, 18) 


['video_id' 'trending_date' 'title' 'channel_title' 'category_id'
 'publish_time' 'tags' 'views' 'likes' 'dislikes' 'comment_count'
 'thumbnail_link' 'comments_disabled' 'ratings_disabled'
 'video_error_or_removed' 'description' 'country' 'category'] 

video_id                  object
trending_date             object
title                     object
channel_title             object
category_id                int64
publish_time              object
tags                      object
views                      int64
likes                      int64
dislikes                   int64
comment_count              int64
thumbnail_link            object
comments_disabled           bool
ratings_disabled            bool
video_error_or_removed      bool
description               object
country                   object
category                  object
dtype: object


In [74]:
allData['trending_date'] = pd.to_datetime(allData['trending_date'], format="%y.%d.%m")

allData['publish_datetime']  = pd.to_datetime(allData['publish_time'], utc=True)
allData['publish_date']  = allData['publish_datetime'].dt.date
allData['publish_time']  = allData['publish_datetime'].dt.time


In [75]:
id_cols = ["video_id","publish_datetime","trending_date","title","channel_title","category","country","likes","dislikes","comment_count","views"]
new_col_order = id_cols + allData.columns[ ~allData.columns.isin(id_cols)].tolist()
allData = allData[new_col_order]
allData.head()

,video_id,publish_datetime,trending_date,title,channel_title,category,country,likes,dislikes,comment_count,views,category_id,publish_time,tags,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,publish_date
0,n1WpP7iowLc,2017-11-10 17:00:03+00:00,2017-11-14,Eminem - Walk On Water (Audio) ft. Beyoncé,EminemVEVO,Music,Canada,787425,43420,125882,17158579,10,17:00:03,"Eminem|""Walk""|""On""|""Water""|""Aftermath/Shady/In...",https://i.ytimg.com/vi/n1WpP7iowLc/default.jpg,False,False,False,Eminem's new track Walk on Water ft. Beyoncé i...,2017-11-10
1,0dBIkQ4Mz1M,2017-11-13 17:00:00+00:00,2017-11-14,PLUSH - Bad Unboxing Fan Mail,iDubbbzTV,Comedy,Canada,127794,1688,13030,1014651,23,17:00:00,"plush|""bad unboxing""|""unboxing""|""fan mail""|""id...",https://i.ytimg.com/vi/0dBIkQ4Mz1M/default.jpg,False,False,False,STill got a lot of packages. Probably will las...,2017-11-13
2,5qpjK5DgCt4,2017-11-12 19:05:24+00:00,2017-11-14,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,Comedy,Canada,146035,5339,8181,3191434,23,19:05:24,"racist superman|""rudy""|""mancuso""|""king""|""bach""...",https://i.ytimg.com/vi/5qpjK5DgCt4/default.jpg,False,False,False,WATCH MY PREVIOUS VIDEO ▶ \n\nSUBSCRIBE ► http...,2017-11-12
3,d380meD0W0M,2017-11-12 18:01:41+00:00,2017-11-14,I Dare You: GOING BALD!?,nigahiga,Entertainment,Canada,132239,1989,17518,2095828,24,18:01:41,"ryan|""higa""|""higatv""|""nigahiga""|""i dare you""|""...",https://i.ytimg.com/vi/d380meD0W0M/default.jpg,False,False,False,I know it's been a while since we did this sho...,2017-11-12
4,2Vv-BfVoq4g,2017-11-09 11:04:14+00:00,2017-11-14,Ed Sheeran - Perfect (Official Music Video),Ed Sheeran,Music,Canada,1634130,21082,85067,33523622,10,11:04:14,"edsheeran|""ed sheeran""|""acoustic""|""live""|""cove...",https://i.ytimg.com/vi/2Vv-BfVoq4g/default.jpg,False,False,False,🎧: https://ad.gt/yt-perfect\n💰: https://atlant...,2017-11-09


In [82]:
clean_idx = allData[allData.title!="Deleted video"].\
            sort_values(["video_id","country","publish_datetime","trending_date"]).\
            groupby(["video_id","country"])["publish_datetime"].\
            idxmax().values
clData = allData[allData.index.isin(clean_idx)]


,video_id,trend_rep,country
15412,0yE2ORkLxoA,1,Canada
10257,4PD8dRaM8Uc,1,Canada
15680,BKHNOxqsSiM,1,Canada
10547,LJsbf7KNVCA,1,Canada
15318,SOeGkrOx0iA,1,Canada
15415,UCSmH6OLPC4,1,Canada
16487,ai0_hAYmSzY,1,Canada
4373,bUjlYfYDoeA,1,Canada
15719,nx0RGHtP15U,1,Canada
12011,7qmSwj8Pwys,2,Canada


In [ ]:
# #No. of tags:
# allData['n_tags'] = allData['tags'].apply(lambda x: len(x.split("|")) )

# #No. of Repeats:
# video_n  = allData.groupby('video_id')['video_id'].count()
# vid_n_DF = pd.DataFrame({"video_id" : video_n.index,
#                          "trend_rep": video_counts.values})
# allData = pd.merge(allData, vid_n_DF, on="video_id", how="left")
# allData.shape